# AQI Prediction with Scikit-Learn

This notebook uses the canonical hourly Open-Meteo table to model `european_aqi` as a next-hour prediction target. It keeps the workflow teaching-oriented:

- build a leakage-safe chronological split
- compare regression baselines for next-hour AQI
- derive AQI bands and compare simple classifiers
- inspect a short prediction window visually

The notebook reuses the shared project loader from `src/open_meteo.py`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.cross_decomposition import PLSRegression
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor, BaggingClassifier, BaggingRegressor, ExtraTreesClassifier, ExtraTreesRegressor, HistGradientBoostingClassifier, HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.linear_model import BayesianRidge, ElasticNet, HuberRegressor, Lasso, LogisticRegression, QuantileRegressor, Ridge, RidgeClassifier, SGDClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from open_meteo import DEFAULT_DATA_DIR, build_hourly_table

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## Load the canonical hourly table

In [ ]:
table = build_hourly_table(DEFAULT_DATA_DIR)
column_arrays = {name: np.asarray(values, dtype=float) for name, values in table.columns.items()}
timestamps = np.asarray(table.timestamps)

required_columns = [
    'european_air_quality_index__european_aqi',
    'hourly_air_quality_vars__pm2_5',
    'hourly_air_quality_vars__pm10',
    'hourly_air_quality_vars__nitrogen_dioxide',
    'hourly_air_quality_vars__ozone',
    'hourly_air_quality_vars__sulphur_dioxide',
    'hourly_air_quality_vars__carbon_monoxide',
    'misc__temperature_2m',
    'misc__relative_humidity_2m',
    'misc__pressure_msl',
    'misc__precipitation',
    'wind__wind_speed_10m',
    'cloud_cover__cloud_cover',
    'solar_radiation_vars__shortwave_radiation',
    'additional_vars_and_options__is_day',
    'air_quality_us_index__us_aqi',
]
missing = [name for name in required_columns if name not in column_arrays]
if missing:
    raise ValueError(f'Missing expected columns: {missing}')

print(f'Rows: {len(timestamps):,}')
print(f'Columns: {len(column_arrays):,}')
print(f'Start: {timestamps[0]}')
print(f'End:   {timestamps[-1]}')

## Build supervised features for next-hour AQI

We use current air-quality and weather signals, cyclic calendar features, and lagged pollutant terms to predict the next hour of `european_aqi`.

In [ ]:
def build_calendar_features(time_values: np.ndarray) -> dict[str, np.ndarray]:
    hours = np.asarray([int(value[11:13]) for value in time_values], dtype=float)
    months = np.asarray([int(value[5:7]) for value in time_values], dtype=float)
    day_of_year = np.asarray([
        int((np.datetime64(value[:10]) - np.datetime64(value[:4] + '-01-01')) / np.timedelta64(1, 'D')) + 1
        for value in time_values
    ], dtype=float)
    return {
        'hour': hours,
        'month': months,
        'day_of_year': day_of_year,
        'hour_sin': np.sin(2.0 * np.pi * hours / 24.0),
        'hour_cos': np.cos(2.0 * np.pi * hours / 24.0),
        'day_of_year_sin': np.sin(2.0 * np.pi * day_of_year / 366.0),
        'day_of_year_cos': np.cos(2.0 * np.pi * day_of_year / 366.0),
    }

calendar_features = build_calendar_features(timestamps)
target_name = 'european_air_quality_index__european_aqi'
base_feature_names = [
    'hourly_air_quality_vars__pm2_5',
    'hourly_air_quality_vars__pm10',
    'hourly_air_quality_vars__nitrogen_dioxide',
    'hourly_air_quality_vars__ozone',
    'hourly_air_quality_vars__sulphur_dioxide',
    'hourly_air_quality_vars__carbon_monoxide',
    'misc__temperature_2m',
    'misc__relative_humidity_2m',
    'misc__pressure_msl',
    'misc__precipitation',
    'wind__wind_speed_10m',
    'cloud_cover__cloud_cover',
    'solar_radiation_vars__shortwave_radiation',
    'additional_vars_and_options__is_day',
    'air_quality_us_index__us_aqi',
]
lag_feature_names = [
    'european_air_quality_index__european_aqi',
    'hourly_air_quality_vars__pm2_5',
    'hourly_air_quality_vars__pm10',
    'hourly_air_quality_vars__nitrogen_dioxide',
]
lag_hours = [1, 3, 6, 24]
forecast_horizon = 1
max_lag = max(lag_hours)
row_start = max_lag
row_end = len(timestamps) - forecast_horizon

feature_blocks = []
feature_names = []

for name in base_feature_names:
    feature_blocks.append(column_arrays[name][row_start:row_end])
    feature_names.append(name)

for name, values in calendar_features.items():
    feature_blocks.append(values[row_start:row_end])
    feature_names.append(name)

for base_name in lag_feature_names:
    for lag in lag_hours:
        feature_blocks.append(column_arrays[base_name][row_start - lag:row_end - lag])
        feature_names.append(f'{base_name}__lag_{lag}h')

X = np.column_stack(feature_blocks)
y = column_arrays[target_name][row_start + forecast_horizon:row_end + forecast_horizon]
time_index = timestamps[row_start:row_end]

valid_mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
X = X[valid_mask]
y = y[valid_mask]
time_index = time_index[valid_mask]

print(f'Supervised rows: {len(y):,}')
print(f'Feature count: {X.shape[1]}')

In [ ]:
def chronological_split(X: np.ndarray, y: np.ndarray, time_index: np.ndarray, train_ratio: float = 0.7, valid_ratio: float = 0.15):
    total = len(y)
    train_end = int(total * train_ratio)
    valid_end = int(total * (train_ratio + valid_ratio))
    return {
        'X_train': X[:train_end],
        'y_train': y[:train_end],
        't_train': time_index[:train_end],
        'X_valid': X[train_end:valid_end],
        'y_valid': y[train_end:valid_end],
        't_valid': time_index[train_end:valid_end],
        'X_test': X[valid_end:],
        'y_test': y[valid_end:],
        't_test': time_index[valid_end:],
    }

split = chronological_split(X, y, time_index)
for name in ['train', 'valid', 'test']:
    print(name, split[f'X_{name}'].shape, split[f't_{name}'][0], split[f't_{name}'][-1])

## Regression baselines for next-hour `european_aqi`

In [ ]:
def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    return {
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'rmse': float(mean_squared_error(y_true, y_pred, squared=False)),
        'r2': float(r2_score(y_true, y_pred)),
    }

regressors = {
    'ridge': Pipeline([('scale', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'lasso': Pipeline([('scale', StandardScaler()), ('model', Lasso(alpha=0.001, max_iter=5000))]),
    'bayesian_ridge': Pipeline([('scale', StandardScaler()), ('model', BayesianRidge())]),
    'huber': Pipeline([('scale', StandardScaler()), ('model', HuberRegressor())]),
    'quantile_regressor': Pipeline([('scale', StandardScaler()), ('model', QuantileRegressor(quantile=0.5, alpha=0.0, solver='highs'))]),
    'elastic_net': Pipeline([('scale', StandardScaler()), ('model', ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=5000))]),
    'pls_regression': Pipeline([('scale', StandardScaler()), ('model', PLSRegression(n_components=8))]),
    'random_forest': RandomForestRegressor(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=4,
        random_state=42,
        n_jobs=-1,
    ),
    'extra_trees': ExtraTreesRegressor(
        n_estimators=250,
        max_depth=18,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
    'hist_gradient_boosting': HistGradientBoostingRegressor(
        learning_rate=0.06,
        max_depth=10,
        max_iter=250,
        random_state=42,
    ),
    'adaboost_regressor': AdaBoostRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
    ),
    'bagging_regressor': BaggingRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    'knn_regressor': Pipeline([('scale', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=12, weights='distance'))]),
}

regression_rows = []
test_predictions = {}

for model_name, model in regressors.items():
    model.fit(split['X_train'], split['y_train'])
    valid_pred = model.predict(split['X_valid'])
    test_pred = model.predict(split['X_test'])
    valid_scores = regression_metrics(split['y_valid'], valid_pred)
    test_scores = regression_metrics(split['y_test'], test_pred)
    regression_rows.append({
        'model': model_name,
        'valid_mae': valid_scores['mae'],
        'valid_rmse': valid_scores['rmse'],
        'test_mae': test_scores['mae'],
        'test_rmse': test_scores['rmse'],
        'test_r2': test_scores['r2'],
    })
    test_predictions[model_name] = test_pred

gpr_train_limit = min(2500, len(split['y_train']))
gpr_eval_limit = min(1500, len(split['y_test']), len(split['y_valid']))
gpr = Pipeline([
    ('scale', StandardScaler()),
    ('model', GaussianProcessRegressor(
        kernel=RBF(length_scale=2.0) + WhiteKernel(noise_level=1.0),
        alpha=1e-3,
        normalize_y=True,
        random_state=42,
    )),
])
gpr.fit(split['X_train'][:gpr_train_limit], split['y_train'][:gpr_train_limit])
gpr_valid_pred = gpr.predict(split['X_valid'][:gpr_eval_limit])
gpr_test_pred = gpr.predict(split['X_test'][:gpr_eval_limit])
gpr_valid_scores = regression_metrics(split['y_valid'][:gpr_eval_limit], gpr_valid_pred)
gpr_test_scores = regression_metrics(split['y_test'][:gpr_eval_limit], gpr_test_pred)
regression_rows.append({
    'model': 'gaussian_process_subset',
    'valid_mae': gpr_valid_scores['mae'],
    'valid_rmse': gpr_valid_scores['rmse'],
    'test_mae': gpr_test_scores['mae'],
    'test_rmse': gpr_test_scores['rmse'],
    'test_r2': gpr_test_scores['r2'],
})
test_predictions['gaussian_process_subset'] = gpr_test_pred

for row in sorted(regression_rows, key=lambda item: item['test_rmse']):
    print(row)

In [ ]:
best_regressor_name = min((row for row in regression_rows if row['model'] != 'gaussian_process_subset'), key=lambda item: item['test_rmse'])['model']
plot_count = 14 * 24
fig, ax = plt.subplots()
ax.plot(split['t_test'][:plot_count], split['y_test'][:plot_count], label='actual', linewidth=2)
ax.plot(split['t_test'][:plot_count], test_predictions[best_regressor_name][:plot_count], label=best_regressor_name, linewidth=1.8)
ax.set_title('Next-hour European AQI forecast on the first two test weeks')
ax.set_xlabel('Timestamp')
ax.set_ylabel('European AQI')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()

## Turn AQI into categories and compare classifiers

European AQI is ordinal, so a useful companion task is band classification. These coarse bins keep the example simple and interpretable.

In [ ]:
band_edges = np.asarray([20, 40, 60, 80, 100], dtype=float)
band_labels = ['very_good', 'good', 'moderate', 'poor', 'very_poor', 'extremely_poor']

y_band = np.digitize(y, band_edges, right=True)
split_cls = chronological_split(X, y_band, time_index)

classifiers = {
    'logistic_regression': Pipeline([
        ('scale', StandardScaler()),
        ('model', LogisticRegression(max_iter=300, multi_class='auto')),
    ]),
    'linear_svc': Pipeline([
        ('scale', StandardScaler()),
        ('model', LinearSVC(C=1.0, max_iter=5000, dual=False)),
    ]),
    'ridge_classifier': Pipeline([
        ('scale', StandardScaler()),
        ('model', RidgeClassifier(alpha=1.0)),
    ]),
    'sgd_classifier': Pipeline([
        ('scale', StandardScaler()),
        ('model', SGDClassifier(loss='log_loss', alpha=0.0001, max_iter=2000, tol=1e-3, random_state=42)),
    ]),
    'random_forest_classifier': RandomForestClassifier(
        n_estimators=200,
        max_depth=16,
        min_samples_leaf=3,
        random_state=42,
        n_jobs=-1,
    ),
    'extra_trees_classifier': ExtraTreesClassifier(
        n_estimators=250,
        max_depth=18,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
    ),
    'hist_gradient_boosting_classifier': HistGradientBoostingClassifier(
        learning_rate=0.06,
        max_depth=8,
        max_iter=220,
        random_state=42,
    ),
    'adaboost_classifier': AdaBoostClassifier(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
    ),
    'bagging_classifier': BaggingClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
    'knn_classifier': Pipeline([('scale', StandardScaler()), ('model', KNeighborsClassifier(n_neighbors=12, weights='distance'))]),
    'gaussian_nb': Pipeline([('scale', StandardScaler()), ('model', GaussianNB())]),
    'mlp_classifier': Pipeline([('scale', StandardScaler()), ('model', MLPClassifier(hidden_layer_sizes=(64, 32), alpha=0.0005, learning_rate_init=0.001, max_iter=300, early_stopping=True, random_state=42))]),
}

classification_rows = []
best_classifier_name = None
best_classifier_pred = None
best_classifier_score = -1.0

for model_name, model in classifiers.items():
    model.fit(split_cls['X_train'], split_cls['y_train'])
    test_pred = model.predict(split_cls['X_test'])
    weighted_f1 = float(f1_score(split_cls['y_test'], test_pred, average='weighted'))
    accuracy = float(accuracy_score(split_cls['y_test'], test_pred))
    classification_rows.append({
        'model': model_name,
        'accuracy': accuracy,
        'weighted_f1': weighted_f1,
    })
    if weighted_f1 > best_classifier_score:
        best_classifier_score = weighted_f1
        best_classifier_name = model_name
        best_classifier_pred = test_pred

svc_train_limit = min(10000, len(split_cls['y_train']))
svc_test_limit = min(5000, len(split_cls['y_test']))
svc = Pipeline([
    ('scale', StandardScaler()),
    ('model', SVC(C=3.0, gamma='scale')),
])
svc.fit(split_cls['X_train'][:svc_train_limit], split_cls['y_train'][:svc_train_limit])
svc_test_pred = svc.predict(split_cls['X_test'][:svc_test_limit])
classification_rows.append({
    'model': 'svc_subset',
    'accuracy': float(accuracy_score(split_cls['y_test'][:svc_test_limit], svc_test_pred)),
    'weighted_f1': float(f1_score(split_cls['y_test'][:svc_test_limit], svc_test_pred, average='weighted')),
})
for row in sorted(classification_rows, key=lambda item: item['weighted_f1'], reverse=True):
    print(row)

In [ ]:
matrix = confusion_matrix(split_cls['y_test'], best_classifier_pred, labels=np.arange(len(band_labels)))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title(f'Confusion matrix for {best_classifier_name}')
ax.set_xlabel('Predicted band')
ax.set_ylabel('True band')
ax.set_xticklabels(band_labels, rotation=45, ha='right')
ax.set_yticklabels(band_labels, rotation=0)
plt.tight_layout()

## Suggested follow-up experiments

- predict `us_aqi` in parallel and compare transfer between AQI systems
- add rolling pollutant averages and finite-difference features
- build multi-step AQI forecasting targets such as 6-hour and 24-hour horizons
- compare these tabular baselines with sequence models later, while keeping the same chronological split